In [ ]:
"""well today we are going to learn about how to 
1.using the cython:（将python代码编译成c语言来进行运行，这和之前学的pandas coding 的optimization一致，
属于第四个方法。
%%cython,
cdef and cpdef
cdef，定义，一般在定义的时候前边加上那么一句东西，只有c coding能进行调用

cpdef,c和python都可以调用

well，cython中还有着装饰器的作用，和set_up.py等等

anyway，pandas中的内置方法其实底层就是c，调用其内置的方法其实就是在用cython

2.pandas 进行向量化操作的核心，直接调用列，而不是使用行，
（1）：用np.where(condition,value_if_true,value_if_false)来代替if else
（2）：用np.select(conditions,choices)来代替多重，if——elif
返回的是numpy数组，np.ndarray
well,lets turn back to the risk management,the.shift() is the most important function in finance
df['close'].shift(1)表示计算前一天的收盘价，df['close'].shift(20)表示前20天的收盘价


"""

"""
1.shift()
2.np.where()
3.np.select()
4.%%timeit用来进行测试
"""

In [ ]:
"""
import pandas as pd
import numpy as np

# 使用前两周的数据管道生成模拟数据
np.random.seed(42)
dates = pd.date_range('2024-01-01', periods=252, freq='B')
tickers = ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'META']

# 生成价格数据
prices = {}
for ticker in tickers:
    returns = np.random.randn(252) * 0.015
    prices[ticker] = 100 * np.exp(np.cumsum(returns))

df_close = pd.DataFrame(prices, index=dates)
df_returns = df_close.pct_change()
#因为这里只是现实数据，并没有涉及计算，所以不需要dropna()

print("价格矩阵形状:", df_close.shape)
df_close.head()
"""



"""
# 【任务 1】计算每只股票的 20 日动量因子（Momentum）
# 公式：MOM_20 = (当日收盘价 / 20日前收盘价) - 1
# 要求：不使用任何循环，纯向量化操作
# 提示：使用 shift(20)
# --- 你的代码 ---

# 答案对照区（写完后再看）
# mom_20 = df_close / df_close.shift(20) - 1
# mom_20.tail()
"""

"""
# 【任务 2】计算每只股票的 20 日波动率因子
# 公式：过去20个交易日收益率的标准差（年化）
# 要求：一行代码完成
# --- 你的代码 ---

# vol_20 = df_returns.rolling(20).std(ddof=0) * np.sqrt(252)
# vol_20.tail()
"""


"""
# 【任务 3】生成交易信号：动量 > 0.05 做多，动量 < -0.05 做空，否则空仓
# 要求：使用 np.where 实现
# 信号值：1=做多，-1=做空，0=空仓
# --- 你的代码 ---

# signal = np.where(mom_20 > 0.05, 1, np.where(mom_20 < -0.05, -1, 0))
# signal = pd.DataFrame(signal, index=mom_20.index, columns=mom_20.columns)
# signal.tail()
"""


""" 
# 【任务 4】使用 np.select 重写任务3（多重条件更清晰的写法）
# 要求：conditions 和 choices 列表分开定义
# --- 你的代码 ---

# conditions = [
#     mom_20 > 0.05,
#     mom_20 < -0.05,
#     (mom_20 >= -0.05) & (mom_20 <= 0.05)
# ]
# choices = [1, -1, 0]
# signal_v2 = pd.DataFrame(
#     np.select(conditions, choices, default=0),
#     index=mom_20.index,
#     columns=mom_20.columns
# )
# signal_v2.tail()
"""


"""# 【任务 5】性能对比实验：iterrows vs 向量化
# 目标：计算 20 日动量，分别用两种方法，对比时间
# 注意：数据量小的时候差异不大，把数据扩展到 1000 行再试
# --- 你的代码 ---

# 扩展数据到 1000 行
# df_close_large = pd.concat([df_close] * 4, ignore_index=True)

# %%timeit
# # 向量化方法
# mom_vectorized = df_close_large / df_close_large.shift(20) - 1

# %%timeit
# # iterrows 方法（会非常慢）
# def calc_mom_iterrows(df):
#     result = pd.DataFrame(index=df.index, columns=df.columns)
#     for i, row in df.iterrows():
#         if i >= 20:
#             # 这里逻辑复杂，性能灾难
#             pass
#     return result

# 记录速度差异：向量化是 iterrows 的 _____ 倍 """

In [ ]:
"""
Factor Backtest Module
Author: [westli]
Date: 2026-04-24
Description: Vectorized factor calculation and strategy backtesting
"""

import pandas as pd
import numpy as np
from typing import Optional, Dict, Callable
#简单明了地说，callable表示在这里放一个函数

class FactorBacktest:
    """
    向量化因子回测框架
    - 因子计算（动量、波动率、均线交叉等）
    - 信号生成（多空阈值）
    - 策略收益计算
    - 回测绩效评估
    """
    
    def __init__(self, prices: pd.DataFrame):
        """
        parameters:
        prices: DataFrame, 行=日期, 列=资产, 值=收盘价
        """
        self.prices = prices
        self.returns = prices.pct_change()
        self.signals = None
        self.strategy_returns = None
        
    def momentum_factor(self, window: int = 20) -> pd.DataFrame:
        """
        计算动量因子
        window: 回溯窗口（交易日）
        返回: DataFrame，值 = (当前价 / N日前价) - 1
        """
        return self.prices / self.prices.shift(window) - 1
    
    def volatility_factor(self, window: int = 20) -> pd.DataFrame:
        """
        计算波动率因子
        返回: 年化波动率
        """
        return self.returns.rolling(window).std(ddof=0) * np.sqrt(252)
    
    def ma_crossover_factor(self, fast: int = 5, slow: int = 20) -> pd.DataFrame:
        """
        均线交叉因子
        返回: 快线 / 慢线 - 1（正值表示快线在上）
        """
        ma_fast = self.prices.rolling(fast).mean()
        ma_slow = self.prices.rolling(slow).mean()
        return ma_fast / ma_slow - 1
    
    def generate_signals(self, factor: pd.DataFrame, 
                         long_threshold: float = 0.0, 
                         short_threshold: Optional[float] = None) -> pd.DataFrame:
        """
        根据因子值生成交易信号
        factor: 因子值 DataFrame
        long_threshold: 做多阈值（因子大于此值做多）
        short_threshold: 做空阈值（因子小于此值做空），None 表示不做空
        """
        if short_threshold is None:
            short_threshold = -long_threshold
        
        conditions = [
            factor > long_threshold,
            factor < short_threshold
        ]
        choices = [1, -1]
        
        self.signals = pd.DataFrame(
            np.select(conditions, choices, default=0),
            index=factor.index,
            columns=factor.columns
        )
        return self.signals
    
    def calculate_strategy_returns(self, signals: Optional[pd.DataFrame] = None) -> pd.DataFrame:
        """
        计算策略收益率
        策略收益 = 信号(t-1) * 实际收益(t)  # 用前一天的信号，因为今天开盘前决定仓位
        """
        if signals is not None:
            self.signals = signals
        
        if self.signals is None:
            raise ValueError("请先调用 generate_signals() 或传入 signals")
        
        # 关键：信号滞后一天（今天收盘的信号，明天开盘才能交易）
        self.strategy_returns = self.signals.shift(1) * self.returns
        return self.strategy_returns
    
    def calculate_performance(self) -> Dict:
        """
        计算回测绩效指标
        返回: 字典，包含年化收益、波动率、夏普、最大回撤等
        """
        if self.strategy_returns is None:
            raise ValueError("请先调用 calculate_strategy_returns()")
        
        # 等权组合收益率
        portfolio_returns = self.strategy_returns.mean(axis=1)
        
        # 累积净值
        cum_returns = (1 + portfolio_returns).cumprod()
        
        # 年化收益率
        total_return = cum_returns.iloc[-1] - 1
        n_years = len(portfolio_returns) / 252
        annual_return = (1 + total_return) ** (1 / n_years) - 1
        
        # 年化波动率
        annual_vol = portfolio_returns.std(ddof=0) * np.sqrt(252)
        
        # 夏普比率（假设无风险利率为0）
        sharpe = annual_return / annual_vol if annual_vol > 0 else 0
        
        # 最大回撤
        cummax = cum_returns.cummax()
        drawdowns = (cum_returns - cummax) / cummax
        max_drawdown = drawdowns.min()
        
        # 胜率
        win_rate = (portfolio_returns > 0).sum() / portfolio_returns.count()
        
        # 盈亏比
        avg_win = portfolio_returns[portfolio_returns > 0].mean()
        avg_loss = portfolio_returns[portfolio_returns < 0].mean()
        profit_loss_ratio = abs(avg_win / avg_loss) if avg_loss != 0 else np.inf
        
        performance = {
            'annual_return': annual_return,
            'annual_volatility': annual_vol,
            'sharpe_ratio': sharpe,
            'max_drawdown': max_drawdown,
            'win_rate': win_rate,
            'profit_loss_ratio': profit_loss_ratio,
            'total_return': total_return,
            'cum_returns': cum_returns
        }
        
        return performance